from mpl_toolkits.mplot3d import Axes3D
# 문제 5 — 4x4 동차변환 모듈과 최소자승법

회전과 병진을 **한 행렬로 묶는** 것이 동차변환입니다.

$$T=\begin{bmatrix}R & \mathbf{t}\\ \mathbf{0}^{\mathsf{T}} & 1\end{bmatrix}\in\mathbb{R}^{4\times 4}$$

이렇게 묶으면 여러 좌표계를 지나는 변환을 **행렬 곱 하나로 연결**할 수 있습니다(문제 6).

## 이 노트북에서 해야 할 일

| # | 할 일 | 구현할 함수 |
|---|---|---|
| 5-1 | `make_T`, `inv_T` 를 만들고 ① 곱하면 단위행렬 ② 일반 역행렬과 일치 두 가지로 검증 | `make_T`, `inv_T` |
| 5-2 | **점(w=1)과 방향(w=0)의 차이**를 확인하고 왜 그런지 설명 | `to_homogeneous`, `transform_point`, `transform_direction`, `transform_points` |
| 5-3 | 두 변환의 **합성 순서**가 바뀌면 결과가 달라짐을 확인하고 3D 로 나란히 비교 (그림 코드 제공) | — |
| 5-4 | `inv_T` 와 일반 역행렬의 **실행 시간 비교**, 차이를 연산량 관점에서 설명 | `inv_T_batch` |
| 5-5 | **최소자승법** — 과결정 캘리브레이션 문제를 정규방정식으로 풀고 `lstsq` 와 비교, 잔차 정량 평가 | `least_squares_normal_equation`, `rmse` |

> `inv_T` 는 **일반 역행렬 함수를 쓰지 말고** 회전 부분의 전치를 이용한 공식으로 구현합니다.
> `tests/test_transform.py` 도 함께 작성해 제출합니다.
> `# --- 검증 ---` 셀과 그림 셀은 제공된 코드입니다. 참조하는 변수 이름을 맞춰 주세요.

In [1]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.rotation import rot_x, rot_y, rot_z
from src.transform import (inv_T, inv_T_batch, least_squares_normal_equation, make_T, rmse,
                           to_homogeneous, transform_direction, transform_point,
                           transform_points)
from src.vectors import det

rng = np.random.default_rng(42)
np.set_printoptions(precision=6, suppress=True)

for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
    if _f in {f.name for f in __import__("matplotlib").font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _f
        break
plt.rcParams["axes.unicode_minus"] = False


def check(label, condition):
    tag = "PASS" if condition else "FAIL"
    print("[" + tag + "] " + label)
    return bool(condition)


# --- 그림 헬퍼 (그대로 쓰면 됩니다) -----------------------------------------

def draw_frame(ax, T, scale=0.35, alpha=1.0, name=""):
    """동차변환 T 가 나타내는 좌표계를 그린다."""
    o = T[:3, 3]
    colors = ["r", "g", "b"]
    for i in range(3):
        v = T[:3, i] * scale
        ax.quiver(*o, *v, color=colors[i], alpha=alpha, arrow_length_ratio=0.18)
    if name:
        ax.text(*(o + 0.05), name, fontsize=9)


def setup_axes(ax, title, lim=1.0):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(-lim, lim)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(title, fontsize=10)
    ax.set_box_aspect([1, 1, 1])


print("준비 완료")

준비 완료


R = rot_z(np.deg2rad(22.5)) @ rot_y(np.deg2rad(-67.5)) @ rot_x(np.deg2rad(112.5))
t = np.array([0.35, -0.15, 0.55])

T = make_T(R, t)
Ti = inv_T(T)

print("T =
", T)
print("Ti =
", Ti)
print("T @ Ti =
", T @ Ti)
print("np.linalg.inv(T) =
", np.linalg.inv(T))


In [2]:
R = rot_z(np.deg2rad(22.5)) @ rot_y(np.deg2rad(-67.5)) @ rot_x(np.deg2rad(112.5))
t = np.array([0.35, -0.15, 0.55])

# TODO: T = make_T(R, t), Ti = inv_T(T) 를 만들고
#       T, Ti, T @ Ti, np.linalg.inv(T) (# 검산용) 를 출력해 비교하세요.

In [3]:
# --- 검증 --- (제공 코드: 수정하지 마세요)
ok = check("T 의 회전부 == R, 병진부 == t", np.allclose(T[:3, :3], R) and np.allclose(T[:3, 3], t))
ok &= check("① T @ inv_T(T) == I", np.allclose(T @ Ti, np.eye(4)))
ok &= check("① inv_T(T) @ T == I", np.allclose(Ti @ T, np.eye(4)))
ok &= check("② 일반 역행렬 np.linalg.inv 와 일치", np.allclose(Ti, np.linalg.inv(T)))       # 검산용
ok &= check("역변환의 회전부 == R^T (전치 공식)", np.allclose(Ti[:3, :3], R.T))
ok &= check("역변환의 병진부 == -R^T t", np.allclose(Ti[:3, 3], -R.T @ t))
ok &= check("역변환의 회전 부분도 회전행렬 (det = 1)", np.isclose(det(Ti[:3, :3]), 1.0))
ok &= check("마지막 행이 (0,0,0,1)", np.allclose(Ti[3], [0, 0, 0, 1]))
ok &= check("inv_T(inv_T(T)) == T", np.allclose(inv_T(Ti), T))

rand_ok = True
for _ in range(200):
    Rr = rot_z(rng.uniform(-np.pi, np.pi)) @ rot_y(rng.uniform(-np.pi, np.pi)) @ rot_x(rng.uniform(-np.pi, np.pi))
    Tr = make_T(Rr, rng.standard_normal(3))
    rand_ok &= bool(np.allclose(inv_T(Tr) @ Tr, np.eye(4)) and np.allclose(inv_T(Tr), np.linalg.inv(Tr)))
ok &= check("무작위 200개 변환에서 모두 성립", rand_ok)
print("\n5-1 전체 통과:", ok)

NameError: name 'T' is not defined

v = np.array([1.0, 0.0, 0.0])
p1, p2 = np.array([1.0, 2.0, 3.0]), np.array([0.5, -1.0, 2.0])

p_out = transform_point(T, v)
d_out = transform_direction(T, v)

print("p_out (점 변환):", p_out)
print("d_out (방향 변환):", d_out)
print("p_out - d_out:", p_out - d_out)
print("len(p_out):", np.linalg.norm(p_out), "| len(d_out):", np.linalg.norm(d_out))

diff_points = transform_direction(T, p1 - p2)
diff_trans = transform_point(T, p1) - transform_point(T, p2)
print("p1 - p2 방향 변환:", diff_points)
print("변환된 p1 - 변환된 p2:", diff_trans)


In [ ]:
v = np.array([1.0, 0.0, 0.0])
p1, p2 = np.array([1.0, 2.0, 3.0]), np.array([0.5, -1.0, 2.0])

# TODO: p_out = transform_point(T, v), d_out = transform_direction(T, v) 를 출력하고 차이를 확인하세요.
# TODO: 길이가 보존되는 쪽은 어느 쪽인지도 출력해 보세요.
# TODO: 두 점의 차이 (p1 - p2) 를 방향으로 변환한 것과, 변환된 p1 - 변환된 p2 를 비교하세요.

In [ ]:
# --- 검증 --- (제공 코드: 수정하지 마세요)
ok = check("점과 방향의 결과가 다르다", not np.allclose(p_out, d_out))
ok &= check("차이가 정확히 병진 벡터 t", np.allclose(p_out - d_out, t))
ok &= check("방향은 길이가 보존된다", np.isclose(np.linalg.norm(d_out), np.linalg.norm(v)))
ok &= check("방향 변환은 회전만 적용 (R @ v)", np.allclose(d_out, R @ v))
ok &= check("점 변환은 R @ p + t", np.allclose(p_out, R @ v + t))
ok &= check("동차좌표 w 가 1 / 0 으로 만들어진다",
            np.allclose(to_homogeneous(v, 1.0), [1, 0, 0, 1]) and np.allclose(to_homogeneous(v, 0.0), [1, 0, 0, 0]))
ok &= check("두 점의 차이는 방향처럼 변환된다",
            np.allclose(transform_direction(T, p1 - p2),
                        transform_point(T, p1) - transform_point(T, p2)))
ok &= check("원점(0,0,0)을 점으로 변환하면 t", np.allclose(transform_point(T, [0, 0, 0]), t))
ok &= check("영벡터를 방향으로 변환하면 영벡터", np.allclose(transform_direction(T, [0, 0, 0]), 0.0))
ok &= check("transform_points 는 (N,3) 을 한 번에 처리한다",
            np.allclose(transform_points(T, np.stack([v, p1, p2])),
                        np.stack([transform_point(T, q) for q in (v, p1, p2)])))
print("\n5-2 전체 통과:", ok)

T1 = make_T(rot_z(np.deg2rad(67.5)), [0.45, 0.25, 0.10])
T2 = make_T(rot_x(np.deg2rad(22.5)), [0.15, 0.05, 0.35])

C12 = T1 @ T2
C21 = T2 @ T1

print("C12:
", C12)
print("C21:
", C21)
print("회전 차이:
", C12[:3, :3] - C21[:3, :3])
print("C12 병진:", C12[:3, 3], "| C21 병진:", C21[:3, 3])


In [ ]:
T1 = make_T(rot_z(np.deg2rad(67.5)), [0.45, 0.25, 0.10])   # z 67.5도 회전 + (0.45, 0.25, 0.10) 이동
T2 = make_T(rot_x(np.deg2rad(22.5)), [0.15, 0.05, 0.35])   # x 22.5도 회전 + (0.15, 0.05, 0.35) 이동

# TODO: C12 = T1 @ T2, C21 = T2 @ T1 을 만들고
#       두 행렬, 회전 부분 차이, 병진 부분을 각각 출력하세요.
# TODO: 전개 공식(회전이 병진에 어떻게 걸리는지)이 맞는지 확인해 출력하세요.
#       예) T1[:3,:3] @ T2[:3,3] + T1[:3,3] 이 C12[:3,3] 과 같은가

In [ ]:
import mpl_toolkits.mplot3d
# --- 3D 그림 (제공 코드) ---
fig = plt.figure(figsize=(13, 4.6))

ax = fig.add_subplot(1, 3, 1, projection="3d")
setup_axes(ax, "기준 좌표계와 T1, T2 각각")
draw_frame(ax, np.eye(4), alpha=0.3, name="base")
draw_frame(ax, T1, name="T1")
draw_frame(ax, T2, name="T2")

ax = fig.add_subplot(1, 3, 2, projection="3d")
setup_axes(ax, "순서 ①: T1 @ T2\n(T2 를 먼저 적용)")
draw_frame(ax, np.eye(4), alpha=0.2, name="base")
draw_frame(ax, C12, name="T1@T2")
ax.plot(*np.array([[0, 0, 0], C12[:3, 3]]).T, "k--", lw=1)

ax = fig.add_subplot(1, 3, 3, projection="3d")
setup_axes(ax, "순서 ②: T2 @ T1\n(T1 을 먼저 적용)")
draw_frame(ax, np.eye(4), alpha=0.2, name="base")
draw_frame(ax, C21, name="T2@T1")
ax.plot(*np.array([[0, 0, 0], C21[:3, 3]]).T, "k--", lw=1)

fig.suptitle("동차변환 합성 순서 비교 — 자세도 위치도 달라진다", fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
# --- 검증 --- (제공 코드: 수정하지 마세요)
ok = check("C12, C21 의 정의가 맞다", np.allclose(C12, T1 @ T2) and np.allclose(C21, T2 @ T1))
ok &= check("두 합성 결과가 다르다", not np.allclose(C12, C21))
ok &= check("병진 부분이 다르다", not np.allclose(C12[:3, 3], C21[:3, 3]))
ok &= check("합성 공식 R1 t2 + t1 이 맞다",
            np.allclose(C12[:3, 3], T1[:3, :3] @ T2[:3, 3] + T1[:3, 3]))
ok &= check("합성 공식 R2 t1 + t2 이 맞다",
            np.allclose(C21[:3, 3], T2[:3, :3] @ T1[:3, 3] + T2[:3, 3]))
ok &= check("두 결과 모두 유효한 동차변환 (회전부 det = 1)",
            np.isclose(det(C12[:3, :3]), 1.0) and np.isclose(det(C21[:3, :3]), 1.0))
ok &= check("역변환 순서는 뒤집힌다: (T1 T2)^-1 == T2^-1 T1^-1",
            np.allclose(inv_T(C12), inv_T(T2) @ inv_T(T1)))
print("\n5-3 전체 통과:", ok)

N_REPEAT = 20000
N_BATCH = 3000

def bench(fn, repeat):
    fn()
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best

rng = np.random.default_rng(42)
Ts_batch = np.array([make_T(rot_z(rng.uniform(-np.pi, np.pi)), rng.standard_normal(3)) for _ in range(N_BATCH)])

t_fast = bench(lambda: inv_T(T), N_REPEAT)
t_generic = bench(lambda: np.linalg.inv(T), N_REPEAT)

t_batch_fast = bench(lambda: inv_T_batch(Ts_batch), 100)
t_batch_generic = bench(lambda: np.array([np.linalg.inv(t) for t in Ts_batch]), 100)

print(f"단건: inv_T = {t_fast * 1e6:.2f} us | np.linalg.inv = {t_generic * 1e6:.2f} us")
print(f"배치 ({N_BATCH}개): inv_T_batch = {t_batch_fast * 1e3:.2f} ms | loop np.linalg.inv = {t_batch_generic * 1e3:.2f} ms")


In [ ]:
N_REPEAT = 20000
N_BATCH = 3000


def bench(fn, repeat):
    """워밍업 후 repeat 번 재서 최솟값을 쓴다. (그대로 쓰면 됩니다)"""
    fn()
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best


# 배치용 변환 묶음 (N_BATCH, 4, 4) — 그대로 쓰면 됩니다
Ts = np.stack([make_T(rot_z(a) @ rot_y(b_) @ rot_x(c), rng.standard_normal(3))
               for a, b_, c in rng.uniform(-np.pi, np.pi, (N_BATCH, 3))])

# TODO: 단건 호출 시간을 재서 출력하세요.
#   t_fast    = bench(lambda: inv_T(T), N_REPEAT)
#   t_generic = bench(lambda: np.linalg.inv(T), N_REPEAT)
# TODO: 배치 시간을 재서 출력하세요.
#   t_batch_fast    = bench(lambda: inv_T_batch(Ts), 200)
#   t_batch_generic = bench(lambda: np.linalg.inv(Ts), 200)
# TODO: 두 방법의 회전부 직교성 오차 max|R^T R - I| 도 비교 출력하세요.

In [ ]:
# --- 검증 --- (제공 코드: 수정하지 마세요)
ok = check("단건 시간이 측정되었다 (양수)", t_fast > 0 and t_generic > 0)
ok &= check("배치에서 inv_T_batch 가 일반 역행렬보다 빠르다", t_batch_fast < t_batch_generic)
ok &= check("inv_T 와 일반 역행렬의 결과가 같다", np.allclose(inv_T(T), np.linalg.inv(T)))       # 검산용
ok &= check("inv_T_batch 결과 == np.linalg.inv 스택 결과",
            np.allclose(inv_T_batch(Ts), np.linalg.inv(Ts)))                                   # 검산용
ok &= check("inv_T_batch 결과 == inv_T 를 하나씩 적용한 결과",
            np.allclose(inv_T_batch(Ts[:50]), np.array([inv_T(t_) for t_ in Ts[:50]])))
ok &= check("inv_T_batch 의 출력 shape 이 (N, 4, 4)", inv_T_batch(Ts).shape == Ts.shape)
ok &= check("inv_T 결과의 회전부가 완전한 직교",
            np.max(np.abs(inv_T(T)[:3, :3].T @ inv_T(T)[:3, :3] - np.eye(3))) < 1e-15)
print("\n5-4 전체 통과:", ok)

N_PTS = 40
NOISE = 2e-3

T_true = make_T(rot_z(np.deg2rad(37.5)) @ rot_x(np.deg2rad(-22.5)), [0.18, -0.42, 0.61])
M_true = T_true[:3, :]

rng = np.random.default_rng(42)
P_cam = rng.uniform(-0.5, 0.5, size=(N_PTS, 3))
P_base = transform_points(T_true, P_cam, w=1.0) + rng.normal(0, NOISE, size=(N_PTS, 3))

A_ls = np.zeros((3 * N_PTS, 12))
b_ls = P_base.reshape(-1)

for i in range(N_PTS):
    x, y, z = P_cam[i]
    A_ls[3*i]     = [x, y, z, 1, 0, 0, 0, 0, 0, 0, 0, 0]
    A_ls[3*i + 1] = [0, 0, 0, 0, x, y, z, 1, 0, 0, 0, 0]
    A_ls[3*i + 2] = [0, 0, 0, 0, 0, 0, 0, 0, x, y, z, 1]

print("A_ls shape:", A_ls.shape, "| b_ls shape:", b_ls.shape)
print("Full rank 여부 (rank == 12):", np.linalg.matrix_rank(A_ls) == 12)

x_hat, residual = least_squares_normal_equation(A_ls, b_ls)
M_hat = x_hat.reshape(3, 4)

x_ref, _, _, _ = np.linalg.lstsq(A_ls, b_ls, rcond=None)
print("x_hat == x_ref:", np.allclose(x_hat, x_ref))


In [ ]:
N_PTS = 40
NOISE = 2e-3          # 2 mm 측정 노이즈

T_true = make_T(rot_z(np.deg2rad(37.5)) @ rot_x(np.deg2rad(-22.5)), [0.18, -0.42, 0.61])
M_true = T_true[:3, :]                                   # 참값 3x4

# TODO: 대응점 P_cam (N_PTS, 3) 을 rng.uniform(-0.5, 0.5, ...) 로 만들고,
#       P_base = transform_points(T_true, P_cam) + NOISE * rng.standard_normal(...) 을 만드세요.
# TODO: 설계행렬 A_ls (3N x 12) 와 관측 b_ls (3N,) 를 조립하고 shape / 과결정 여부 / rank 를 출력하세요.
#       힌트: Ph = to_homogeneous(P_cam, 1.0);  각 행 row 에 대해 np.kron(np.eye(3), row.reshape(1, 4))
# TODO: x_hat, residual = least_squares_normal_equation(A_ls, b_ls);  M_hat = x_hat.reshape(3, 4)
# TODO: x_ref = np.linalg.lstsq(A_ls, b_ls, rcond=None)[0]  (# 비교 대상) 와 비교하고
#       rmse_val = rmse(residual), |A_ls.T @ residual|, R_hat = M_hat[:3, :3] 의 det 를 출력하세요.

In [ ]:
# --- 그래프 (제공 코드) ---
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))

pred = (A_ls @ x_hat).reshape(-1, 3)
err_per_point = np.linalg.norm(P_base - pred, axis=1)

axes[0].hist(residual * 1e3, bins=20, color="steelblue", edgecolor="white")
axes[0].axvline(0, color="k", lw=1)
axes[0].set_xlabel("잔차 [mm]")
axes[0].set_ylabel("빈도")
axes[0].set_title("잔차 분포 (RMSE = {:.4f} mm)".format(rmse_val * 1e3))
axes[0].grid(alpha=0.3, axis="y")

axes[1].bar(np.arange(N_PTS), err_per_point * 1e3, color="darkorange")
axes[1].axhline(NOISE * 1e3, ls="--", color="k", lw=1, label="주입 노이즈 sigma = {:.0f} mm".format(NOISE * 1e3))
axes[1].set_xlabel("대응점 번호")
axes[1].set_ylabel("점별 오차 [mm]")
axes[1].set_title("대응점별 재투영 오차")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3, axis="y")

fig.tight_layout()
plt.show()

In [ ]:
# --- 검증 --- (제공 코드: 수정하지 마세요)
ok = check("과결정 문제다 (식 120개 > 미지수 12개)", A_ls.shape == (3 * N_PTS, 12) and b_ls.shape == (3 * N_PTS,))
ok &= check("설계행렬이 full column rank", np.linalg.matrix_rank(A_ls) == 12)                  # 검산용
ok &= check("정규방정식 해가 np.linalg.lstsq 와 일치", np.allclose(x_hat, x_ref))              # 비교 대상
ok &= check("잔차 정의가 b - A x 이다", np.allclose(residual, b_ls - A_ls @ x_hat))
ok &= check("잔차가 A 의 열공간에 수직 (A^T r = 0)", np.allclose(A_ls.T @ residual, 0.0, atol=1e-9))
ok &= check("추정한 M 이 참값에 가깝다 (오차 < 6 mm 수준)", np.max(np.abs(M_hat - M_true)) < 6e-3)
ok &= check("잔차 RMSE 가 주입 노이즈와 같은 자릿수", 0.2 * NOISE < rmse_val < 2.0 * NOISE)
ok &= check("추정 회전부의 det 가 1 에 가깝다", abs(det(R_hat) - 1.0) < 2e-2)
ok &= check("최소자승해가 실제로 최소 (임의 교란보다 잔차가 작다)",
            all(np.linalg.norm(b_ls - A_ls @ (x_hat + 1e-3 * rng.standard_normal(12)))
                > np.linalg.norm(residual) for _ in range(20)))
print("\n5-5 전체 통과:", ok)

## 답안 템플릿 정리

In [ ]:
summary = """
1. 역변환 검증: 단위행렬 여부 ___ / 일반 역행렬과 일치 ___
   - 사용한 공식: ___

2. 점 변환 결과: ___ / 방향 변환 결과: ___
   - 차이의 이유: ___
   - 두 결과의 차이가 무엇과 같은가: ___

3. 합성 순서 비교 그림: 위 3분할 그림 참조
   - T1@T2 병진 ___ vs T2@T1 병진 ___
   - 달라지는 이유: ___

4. inv_T 와 일반 역행렬 속도
   - 단건 호출 : ___ us / ___ us
   - 배치 ___ 개 : ___ ms / ___ ms
   - 차이의 이유: ___

5. 최소자승 해: ___
   - lstsq 와 일치: ___
   - 잔차 RMSE: ___ m (= ___ mm), 주입 노이즈 sigma = 2.0 mm
   - 잔차가 열공간에 수직임을 보인 값 |A^T r| = ___
"""
print(summary)